# Multimedia Project
Image Color Conversion GUI

Run all cells, upload an image, then press a button to convert it.

Section 1: RGB to Grayscale (Mohamed Hany)

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

In [ ]:
def to_rgb(img):
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)


def to_gray(img):
    return cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)


def to_binary(img, t=128):
    gray = to_gray(img)
    _, binary = cv2.threshold(gray, t, 255, cv2.THRESH_BINARY)
    return binary


def to_cmy(img):
    rgb = to_rgb(img) / 255.0
    r, g, b = cv2.split(rgb)
    c = 1 - r
    m = 1 - g
    y = 1 - b
    return (cv2.merge([c, m, y]) * 255).astype(np.uint8)


def to_cmyk(img):
    rgb = to_rgb(img) / 255.0
    r, g, b = cv2.split(rgb)
    k = 1 - np.maximum(np.maximum(r, g), b)
    c = (1 - r - k) / (1 - k + 1e-10)
    m = (1 - g - k) / (1 - k + 1e-10)
    y = (1 - b - k) / (1 - k + 1e-10)
    return (cv2.merge([c, m, y, k]) * 255).astype(np.uint8)

In [ ]:
title = widgets.HTML("<h2 style='margin:4px 0'>Image Color Converter</h2>")
upload = widgets.FileUpload(accept="image/*", multiple=False, description="Upload Image", button_style="danger", layout=widgets.Layout(width="160px", height="36px"))
slider = widgets.IntSlider(value=128, min=0, max=255, description="Threshold", layout=widgets.Layout(width="400px"))
lay = widgets.Layout(width="110px", height="36px")
b_rgb = widgets.Button(description="RGB", button_style="info", layout=lay)
b_gray = widgets.Button(description="Grayscale", button_style="primary", layout=lay)
b_bin = widgets.Button(description="Binary", button_style="primary", layout=lay)
b_cmy = widgets.Button(description="CMY", button_style="warning", layout=lay)
b_cmyk = widgets.Button(description="CMYK", button_style="success", layout=lay)
out = widgets.Output()

img = None
mode = None


def get_image():
    v = upload.value
    if not v:
        return None
    item = list(v.values())[0] if isinstance(v, dict) else v[0]
    data = np.frombuffer(item["content"], np.uint8)
    return cv2.imdecode(data, cv2.IMREAD_COLOR)


def show(name):
    global img, mode
    mode = name
    img = get_image()
    with out:
        clear_output(wait=True)
        if img is None:
            print("Upload an image first")
            return
        plt.figure(figsize=(7, 7))
        if name == "RGB":
            plt.imshow(to_rgb(img))
        elif name == "Grayscale":
            plt.imshow(to_gray(img), cmap="gray")
        elif name == "Binary":
            plt.imshow(to_binary(img, slider.value), cmap="gray")
        elif name == "CMY":
            plt.imshow(to_cmy(img))
        elif name == "CMYK":
            res = to_cmyk(img)
            for i, ch in enumerate("CMYK"):
                plt.subplot(2, 2, i + 1)
                plt.imshow(res[:, :, i], cmap="gray")
                plt.title(ch)
                plt.axis("off")
        if name != "CMYK":
            plt.title(name)
            plt.axis("off")
        plt.show()


def on_slider(change):
    if mode == "Binary":
        show("Binary")


upload.observe(lambda c: show("RGB"), names="value")
slider.observe(on_slider, names="value")
b_rgb.on_click(lambda b: show("RGB"))
b_gray.on_click(lambda b: show("Grayscale"))
b_bin.on_click(lambda b: show("Binary"))
b_cmy.on_click(lambda b: show("CMY"))
b_cmyk.on_click(lambda b: show("CMYK"))

display(widgets.VBox([title, upload, widgets.HBox([b_rgb, b_gray, b_bin, b_cmy, b_cmyk]), slider, out]))